## Importing Libraries

In [42]:
from ollama import chat
import glob
from tqdm import tqdm
import os
import json
import re
from difflib import SequenceMatcher
from groq import Groq

## Setting up files

In [43]:
GENERATION_MODEL = "deepseek-r1:8b" # llama3.1:8b, qwen3:8b, deepseek-r1:8b
GROQ_MODEL = "openai/gpt-oss-120b"

GROQ_KEY = os.getenv("GROQ_API_KEY")
CLIENT = Groq(api_key=GROQ_KEY)

TYPE_LLM = False # True - local, False - groq

FILES_CONV = glob.glob("../Test_Files/Clinical_trials/Criteria_extracted/clinical-trial-extracted*.txt")
GOLD_FILES = glob.glob("../Test_Files/Clinical_trials/GT-clinical-trial_*.json")

PROMPT_CONV_FILE = "./prompts/criteria_conversion/criteria-conversion_prompt.txt"
SYS_PROMPT_CONV_FILE = "./prompts/criteria_conversion/sys_criteria-conversion_prompt.txt"

OUTPUT_CONV_DIR = "./llm-outputs/criteria-conversion/"
OUTPUT_CONV_FILE = "experiment"

CHUNK_MAX_CHARS = 1000   
CHUNK_OVERLAP   = 200    
CRITERIA_BATCH  = 5

print(f"Found the following files for conversion - {FILES_CONV}")
print(f"Found the following golden diaries {GOLD_FILES}")

Found the following files for conversion - ['../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e1.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e10.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e11.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e12.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e13.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e14.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e15.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e16.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e17.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e18.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e19.txt', '../Test_Fi

## Setting up environment

In [44]:
## Setting evironment
def set_env(prompt_file, sys_prompt_file,output_dir):
    with open(prompt_file,"r", encoding="utf-8") as p:
        base_prompt = p.read()
        
    with open(sys_prompt_file, "r", encoding = "utf-8") as sp:
        sys_prompt = sp.read()

    os.makedirs(output_dir,exist_ok=True)

    count = 0

    for path in os.listdir(output_dir):
        if os.path.isfile(os.path.join(output_dir, path)):
            count += 1
    
    return base_prompt, sys_prompt, count

base_prompt_conv, sys_prompt_conv, count_conv_exp = set_env(PROMPT_CONV_FILE, SYS_PROMPT_CONV_FILE, OUTPUT_CONV_DIR)

## Helpers

In [45]:
def normalize(text):
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'within \d+ .*', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()
 
 
def is_similar(a, b, threshold=0.92):
    return SequenceMatcher(None, a, b).ratio() > threshold
 
 
def deduplicate(items):
    result = []
    for item in items:
        raw = item.get("text", "") if isinstance(item, dict) else item
        text = normalize(raw)
        if not any(
            is_similar(
                text,
                normalize(r.get("text", "") if isinstance(r, dict) else r)
            )
            for r in result
        ):
            result.append(item)
    return result

def extract_json_from_response(text):
    match = re.search(r"```json\s*([\s\S]*?)```", text)
    if match:
        return json.loads(match.group(1))
 
    match = re.search(r"(\{[\s\S]*\})", text)
    if match:
        return json.loads(match.group(1))
 
    raise ValueError(f"Nenhum JSON encontrado na resposta:\n{text[:500]}")

def merge_results(results):
    merged = {}
    for result in results:
        for key, value in result.items():
            if key not in merged:
                merged[key] = []
            if isinstance(value, list):
                merged[key].extend(value)
            else:
                merged[key] = value
 
    for key in merged:
        if isinstance(merged[key], list):
            merged[key] = deduplicate(merged[key])
 
    return merged


## LLM Call

In [46]:
def call_llm(sys_prompt, user_prompt):
    if TYPE_LLM:
        stream = chat(
            model=GENERATION_MODEL,
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            stream=True,
            options={"num_ctx": 32000},
        )
        return "".join(chunk["message"]["content"] for chunk in stream)
    else:
        response = CLIENT.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=0,
        )
        return response.choices[0].message.content

## Chunking

In [47]:
def split_text_into_chunks(text, max_chars=CHUNK_MAX_CHARS, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        if end < len(text):
            cut = text.rfind(" ", start, end)
            if cut > start:
                end = cut
        chunks.append(text[start:end].strip())
        start = end - overlap
    return chunks

## Prompt run

In [ ]:
def run_single_prompt(sys_prompt, user_prompt, label="", max_retries=3):
    for attempt in range(1, max_retries + 1):
        raw = call_llm(sys_prompt, user_prompt)
        try:
            return extract_json_from_response(raw)
        except (ValueError, json.JSONDecodeError) as e:
            print(f"  [{label}] Try {attempt}/{max_retries} – error JSON: {e}")
            print(f"  Raw output (first 300 chars): {raw[:300]}")
    raise ValueError(f"[{label}] Failed after {max_retries} tries.")

def chunk_criteria_list(criteria_list, batch_size=5):
    for i in range(0, len(criteria_list), batch_size):
        yield criteria_list[i:i + batch_size]


def run_criteria_pipeline(sys_prompt, base_prompt, criteria_json, batch_size=5, label=""):

    all_results = []

    for criteria_type in ["inclusion_criteria", "exclusion_criteria"]:
        criteria_list = criteria_json.get(criteria_type, [])
        if not criteria_list:
            continue

        print(f"  [{label}] {criteria_type}: {len(criteria_list)} criteria → batches of {batch_size}")

        for i, batch in enumerate(chunk_criteria_list(criteria_list, batch_size), 1):
            payload = {
                "inclusion_criteria": batch if criteria_type == "inclusion_criteria" else [],
                "exclusion_criteria": batch if criteria_type == "exclusion_criteria" else [],
            }

            batch_label = f"{label} – {criteria_type} batch {i}"
            prompt = base_prompt.replace("{{CRITERIA_TEXT}}", json.dumps(payload, ensure_ascii=False))

            result = run_single_prompt(sys_prompt, prompt, label=batch_label)
            all_results.append(result)

    return merge_results(all_results)

## Criteria Conversion
In this second phase the already extracted criteria in natural language of a given clinical trial will be converted into logical rules that way allowing the deterministic matching of patients with the clinical trial

In [51]:
count_conv_exp = 7

pbar = tqdm(total=len(FILES_CONV), desc="Processing trials for criteria conversion")

for file in FILES_CONV:
    
    trial_id = int(file.split("_")[-1].split(".")[0].split("e")[-1])
    label = f"trial-{trial_id}"
    if trial_id not in [26,27,28,29,3,30,4,5,6,7,8,9,1,10,11,12,13,14,15,16,17,18,19,2,20,21,22]:
        with open(file,"r", encoding="utf-8") as f:
            text_arr = [t.strip() for t in f.readlines() if t.strip()]
            text = " ".join(text_arr)
            
            print(f"processing file: {file}")
            
            prompt = base_prompt_conv.replace("{{CRITERIA_TEXT}}",text)
        

            if TYPE_LLM:
                stream = chat(
                    model=GENERATION_MODEL,
                    messages=[
                        {
                            "role": "system",
                            "content": sys_prompt_conv
                        },
                        {
                            "role": "user", 
                            "content": prompt
                            }
                        ],
                    stream=True,
                    options={"num_ctx": 32000}
                    )
                
                llm_output = ""
                for chunk in stream:
                    llm_output += chunk["message"]["content"]
                    
            elif not TYPE_LLM:
                criteria_json = json.loads(text)

                result = run_criteria_pipeline(
                    sys_prompt=sys_prompt_conv,
                    base_prompt=base_prompt_conv,
                    criteria_json=criteria_json,
                    batch_size=3,
                    label=label,
                )
                llm_output = json.dumps(result, ensure_ascii=False, indent=2)

            with open(f"{OUTPUT_CONV_DIR}{OUTPUT_CONV_FILE}-{count_conv_exp}.txt","a",encoding="utf-8") as o:
                o.write(f"Ouput for file {file}\n")
                o.write(f"{llm_output}\n\n")
                print(f"Saved LLM output on {OUTPUT_CONV_FILE}-{count_conv_exp}")
                
            
            print("\n")                                                                                                                                                                                                                                                             
            
            pbar.update(1)
    else:
        print(f"skipping trial {trial_id} as it was already processed")
        
pbar.close()

Processing trials for criteria conversion:  50%|█████     | 15/30 [07:53<07:53, 31.55s/it]


skipping trial 1 as it was already processed
skipping trial 10 as it was already processed
skipping trial 11 as it was already processed
skipping trial 12 as it was already processed
skipping trial 13 as it was already processed
skipping trial 14 as it was already processed
skipping trial 15 as it was already processed
skipping trial 16 as it was already processed
skipping trial 17 as it was already processed
skipping trial 18 as it was already processed
skipping trial 19 as it was already processed
skipping trial 2 as it was already processed
skipping trial 20 as it was already processed
skipping trial 21 as it was already processed
skipping trial 22 as it was already processed
processing file: ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extracted_e23.txt
  [trial-23] inclusion_criteria: 34 critérios → batches de 3
  [trial-23] exclusion_criteria: 28 critérios → batches de 3


Saved LLM output on experiment-7


processing file: ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extracted_e24.txt
  [trial-24] inclusion_criteria: 26 critérios → batches de 3
  [trial-24] exclusion_criteria: 13 critérios → batches de 3


Saved LLM output on experiment-7


processing file: ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extracted_e25.txt
  [trial-25] inclusion_criteria: 14 critérios → batches de 3
  [trial-25] exclusion_criteria: 33 critérios → batches de 3


Processing trials for criteria conversion:  10%|█         | 3/30 [03:00<27:00, 60.01s/it]

Saved LLM output on experiment-7


skipping trial 26 as it was already processed
skipping trial 27 as it was already processed
skipping trial 28 as it was already processed
skipping trial 29 as it was already processed
skipping trial 3 as it was already processed
skipping trial 30 as it was already processed
skipping trial 4 as it was already processed
skipping trial 5 as it was already processed
skipping trial 6 as it was already processed
skipping trial 7 as it was already processed
skipping trial 8 as it was already processed
skipping trial 9 as it was already processed


## Evaluation
In this phase the pipeline of extraction will be evaluated in 2 different fields:
- Correct classification (inclusion/exclusion)
- Logic correctness of rules

In [ ]:
for gold_file in GOLD_FILES:
    with open(gold_file,"r",encoding="utf-8") as gf, \
         open(f"{OUTPUT_EXTR_DIR}{OUTPUT_EXTR_FILE}-{1}.txt","r",encoding="utf-8") as out_extr, \
         open(f"{OUTPUT_CONV_DIR}{OUTPUT_CONV_FILE}-{2}.txt","r",encoding="utf-8") as out_conv:
        
        curr_gf_trial = gold_file.split('_')[3].split('.')[0]
        
        print("Current trial: ", curr_gf_trial)

        data_gf = json.load(gf)

        out_extr_arr = [t.strip() for t in out_extr.readlines() if t.strip()]
        out_conv_arr = [t.strip() for t in out_conv.readlines() if t.strip()]

        out_extr_text = " ".join(out_extr_arr)
        out_conv_text = " ".join(out_conv_arr)

        outputs_extraction = out_extr_text.split("Ouput for file ")
        outputs_extraction.pop(0)

        outputs_conversion = out_conv_text.split("Ouput for file ")
        outputs_conversion.pop(0)
        
        print("outputs_extraction: ", outputs_extraction)
        print("outputs_conversion: ", outputs_conversion)
        
        for output_c, output_e in zip(outputs_conversion, outputs_extraction):
            if curr_gf_trial not in output_c or curr_gf_trial not in output_e:
                continue

            # Extract JSON safely
            json_conv_match = re.search(r"\{.*\}", output_c, flags=re.DOTALL)
            if not json_conv_match:
                print("No JSON found for", curr_gf_trial)
                continue
            
            json_extr_match = re.search(r"\{.*\}", output_e, flags=re.DOTALL)
            if not json_extr_match:
                print("No JSON found for", curr_gf_trial)
                continue
            
            output_conv_json = json.loads(json_conv_match.group(0))
            output_extr_json = json.loads(json_extr_match.group(0))

            print("Golden truth data ", data_gf)
            print("Output of the LLM (conversion) ", output_conv_json)
            print("Output of the LLM (extraction) ", output_extr_json)
            
            # Correct classification (inclusion/exclusion)
            
            gf_inclusion = set([c.strip().lower() for c in data_gf["inclusion_criteria"]])
            gf_exclusion = set([c.strip().lower() for c in data_gf["exclusion_criteria"]])

            llm_inclusion = set([c.strip().lower() for c in output_extr_json["inclusion_criteria"]])
            llm_exclusion = set([c.strip().lower() for c in output_extr_json["exclusion_criteria"]])
            
            correct = 0
            total = 0

            for crit in llm_inclusion:
                total += 1
                if any(crit in g or g in crit for g in gf_inclusion):
                    correct += 1

            for crit in llm_exclusion:
                total += 1
                if any(crit in g or g in crit for g in gf_exclusion):
                    correct += 1

            accuracy = correct / total if total > 0 else 0

            print("Correct classification:", correct, "/", total)
            print("Accuracy:", round(accuracy, 4))